# 3. Modify — PDF §7.3, páginas 7–8

Código sin comentarios e identificadores que revelen etiquetas; TF-IDF de tokens y bigramas. El vocabulario se ajusta exclusivamente en train. Los metadatos sirven para auditoría y nunca son características.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Abrir desde avance_semma o su carpeta notebooks'
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown


In [2]:
from src.common import clean_code, code_tokens
from src.experiment import modify
sample = "cursor.execute('SELECT * FROM accounts WHERE id=?', (account_id,))"
print(code_tokens(clean_code(sample)))
matrices, targets = modify()
print({name: value.shape for name, value in matrices.items()})
print('Objetivos:', targets)

['cursor', '.', 'execute', '(', "'", 'select', '*', 'from', 'accounts', 'where', 'id', '=', '?', "'", ',', '(', 'account_id', ',', ')', ')']


{'train': (2001, 2686), 'validation': (338, 2686), 'test': (341, 2686)}
Objetivos: ['vulnerable', 'A01:2025', 'A02:2025', 'A04:2025', 'A05:2025', 'A07:2025', 'API1:2023', 'API2:2023', 'API3:2023', 'API5:2023', 'API8:2023']


In [3]:
display(pd.Series(json.loads((ROOT / 'results/modify_audit.json').read_text(encoding='utf-8'))))
display(pd.read_csv(ROOT / 'data/processed/feature_names.csv').head(30))

fit_partition                                                            train
metadata_used_as_features                                                False
vocabulary_size                                                           2686
shapes                       {'train': [2001, 2686], 'validation': [338, 26...
class_balance_strategy       class_weight=balanced en cada clasificador bin...
targets                      [vulnerable, A01:2025, A02:2025, A04:2025, A05...
input                            code only, comments and benchmark IDs removed
dtype: object

,feature
0,!
1,"! """
2,! doctype
3,! founduser
4,! map
5,!=
6,!= -
7,!= null
8,""""
9,""" """


## Justificación

TF-IDF es una base reproducible para minería de texto/código. Los símbolos y bigramas conservan parte de los patrones de llamadas, operadores y controles. `min_df=2` reduce términos aislados; el límite de 18.000 características controla el costo. Se usa ponderación de clases en cada clasificador binario, sin alterar los conjuntos retenidos. Las matrices se guardan dispersas. Esta representación no entiende flujo de datos, dependencias entre servicios ni lógica completa.